### Structured Output

#### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [15]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3.6-27b", reasoning_effort="none")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x000002086CC142D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002086CC14CD0>, model_name='qwen/qwen3.6-27b', reasoning_effort='none', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [16]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(
        description="The title of the movie"
    )
    year:int=Field(
        description="This year the movie was released"
    )
    director:str=Field(
        description="The director of the movie"
    )
    rating:float=Field(
        description="The movie rating out of 10"
    )

In [18]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x000002086CC142D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002086CC14CD0>, model_name='qwen/qwen3.6-27b', reasoning_effort='none', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 

In [19]:
response = model_with_structure.invoke("Provide me details about the movie Whiplash")
response

Movie(title='Whiplash', year=2014, director='Damien Chazelle', rating=8.5)

### Message Output alongside parsed structure

In [20]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(...,description="The title of the movie")
    year: int = Field(...,description="The year the movie was released")
    director: str = Field(...,description="The director of the movie")
    rating: float = Field(...,description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 's45gqmdtc', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 355, 'total_tokens': 423, 'completion_time': 0.130477609, 'completion_tokens_details': None, 'prompt_time': 0.024971868, 'prompt_tokens_details': None, 'queue_time': 0.051184894, 'total_time': 0.155449477}, 'model_name': 'qwen/qwen3.6-27b', 'system_fingerprint': 'fp_2f860a3fc2', 'service_tier': 'on_demand', 'reasoning_effort': 'none', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0331d-5fd4-75e1-b8b1-158ccf01426a-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.8, 'title': 'Inception', 'year': 2010}, 'id': 's45gqmdtc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 355,

### Nested Structure

In [21]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class Movie_details(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in million USD")

model_with_structure = model.with_structured_output(Movie_details)

response = model_with_structure.invoke("Provide details about movie Inception")
response

Movie_details(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal Cobb'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Michael Caine', role='Miles')], genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160.0)

### TypedDict

TypeDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [23]:
from typing_extensions import TypedDict, Annotated
class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("Please Provide the details for movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [25]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'Avengers',
 'year': 2012}

In [31]:
print(model.profile)

None


### DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator.

In [43]:
import os
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [41]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str=Field(description="The name of the person")
    email: str=Field(description="The email address of the person")
    phone: str=Field(description="The phone number of the person")

agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    response_format=ContactInfo # auto-select ProviderStrategy
)

result = agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "Extract contact info from: John Doe, john@example.com, (555)123-4567"
            }
        ]
    }
)

print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555)123-4567)
result

name='John Doe' email='john@example.com' phone='(555)123-4567'


{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555)123-4567', additional_kwargs={}, response_metadata={}, id='8868527d-2966-4e8e-9bf4-ed5c5601b421'),
  AIMessage(content=[{'type': 'text', 'text': '{"name":"John Doe","email":"john@example.com","phone":"(555)123-4567"}', 'extras': {'signature': 'ErEICq4IARFNMg+u3PNClPynRP6j47+u34dvkeWpCqjPx07DaSvvaq3AnQJDHOO2PnGx2Uyf8C4K5xrQCz9siV0qTgNi/CSE/asxhbFQvPDvi+dkvclm88KyUdjZst190too8q0UOAz7EtooDAclss0P2CPvSQv32XOaVevY6tiUHwRGTPE7GXG6ykNLuqbUYXxrEdLc7bMVu33zxtyC7T0ZDGx44ePOa027pwBy9qG2yckZhsT5oCHtpuNPe7uW0Z9MeL+UFJXlimlFE55yOeI8OnGq4OI+NhL3rE+/0Ox7JQpXkg0x+cjtzqg2EHb3elGHFAw/Nuu7E/NT0JiSBYMazv6Em6L1NsuTB2+AuXhXmiwwfMMN6dFvOVg42Xx5OPkRtR7jxf3ykvE6vXAycBmWGdkfCy+uNzv1h6LWtUDahaC1lVEEDV5C8s7llRMg989jWAGWbAEJuitapd/F7CvNgAk4JdJ2lmJFGnjlD3r4R93SSo7w4Rp5jaEW/JjzxhlP23KDVYENJScG98oefVA9KweHkfttRwLBRHF74WxI5YwyI51Eqlw9FGf7VEE2Z8Ih2UcIf8Dg1BL5Z0LUmcK26QpIeSu/BckvNtWmK/Mmw5XbpTgQ36NaN9GGG3JsWUlyY5yrarKukdPN4beh

In [42]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555)123-4567')

In [46]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person"""
    name: str
    email: str
    phone: str

agent = create_agent(
    model = "google_genai:gemini-3.6-flash",
    response_format = ContactInfo
)

result = agent.invoke(
    {
        "messages":[
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555)123-4567"
            }
        ]
    }
)

result["structured_response"]
# result

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555)123-4567'}

In [47]:
## Dataclass
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person"""
    name: str
    email: str
    phone: str

agent = create_agent(
    model="google_genai:gemini-3.6-flash",
    response_format=ContactInfo
)

result = agent.invoke(
    {
        "messages":[
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555)123-4567"
            }
        ]
    }
)
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555)123-4567')